In [13]:
import sys
import os

sys.path.append(
    os.path.abspath("..")
)

In [14]:
import pandas as pd

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.baseline_models import BaselineModels
from src.evaluation import ModelEvaluator


In [15]:
DATA_PATH = "../data/raw/delivery_data.csv"

preprocessor = DataPreprocessor(DATA_PATH)

engineer = FeatureEngineer()

baseline = BaselineModels()

evaluator = ModelEvaluator()

df = preprocessor.load_data()

df = preprocessor.clean_data(df)

df = engineer.create_temporal_features(df)

df = engineer.create_targets(df)

df = preprocessor.encode_features(df)

Loaded dataset shape: (144867, 24)


In [16]:
train_df, val_df, test_df = (
    preprocessor.split_data(df)
)

print(train_df.shape)
print(test_df.shape)


(100947, 31)
(21632, 31)


In [17]:
linear_model, linear_preds = (
    baseline.train_linear_regression(
        train_df,
        test_df
    )
)

linear_metrics = evaluator.regression_metrics(
    test_df["delay_ratio"],
    linear_preds
)

print("Linear Regression Metrics")

print(linear_metrics)


Linear Regression MAE: 0.5154
Linear Regression Metrics
{'MAE': 0.5154315680453303, 'RMSE': np.float64(0.8493485956927728), 'R2': 0.04563469653467411, 'MAPE': np.float64(26.086105591177144), 'Within_15_Percent': np.float64(42.97337278106509)}


In [18]:
rf_model, rf_preds = (
    baseline.train_random_forest(
        train_df,
        test_df
    )
)

rf_metrics = evaluator.regression_metrics(
    test_df["delay_ratio"],
    rf_preds
)

print("Random Forest Metrics")

print(rf_metrics)

Random Forest MAE: 0.4332
Random Forest Metrics
{'MAE': 0.4331968307198018, 'RMSE': np.float64(0.7587215880780467), 'R2': 0.2384339445465089, 'MAPE': np.float64(20.97376189868721), 'Within_15_Percent': np.float64(54.98335798816568)}


In [19]:
xgb_model, xgb_preds = (
    baseline.train_xgboost(
        train_df,
        test_df
    )
)

xgb_metrics = evaluator.regression_metrics(
    test_df["delay_ratio"],
    xgb_preds
)

print("XGBoost Metrics")

print(xgb_metrics)

XGBoost MAE: 0.4744
XGBoost Metrics
{'MAE': 0.474394328270911, 'RMSE': np.float64(0.789751825476893), 'R2': 0.17486696921796874, 'MAPE': np.float64(23.084062610309847), 'Within_15_Percent': np.float64(45.99667159763314)}


In [20]:
comparison_df = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "MAE": [
        linear_metrics["MAE"],
        rf_metrics["MAE"],
        xgb_metrics["MAE"]
    ],
    "RMSE": [
        linear_metrics["RMSE"],
        rf_metrics["RMSE"],
        xgb_metrics["RMSE"]
    ],
    "MAPE": [
        linear_metrics["MAPE"],
        rf_metrics["MAPE"],
        xgb_metrics["MAPE"]
    ]
})

print(comparison_df)


               Model       MAE      RMSE       MAPE
0  Linear Regression  0.515432  0.849349  26.086106
1      Random Forest  0.433197  0.758722  20.973762
2            XGBoost  0.474394  0.789752  23.084063


In [21]:
predictions_df = pd.DataFrame({
    "actual": test_df["delay_ratio"],
    "predicted": xgb_preds
})

predictions_df.to_csv(
    "../outputs/predictions/xgb_predictions.csv",
    index=False
)

print("Baseline Modeling Completed")

Baseline Modeling Completed
